# AIDA 2156 Final Project – 04 Outliers, Feature Relationships and Sensitivity

**Data version 2** (`shared/synthetic-data/`): the official cleaned requests (cleaning version 1.1), with notebook 02's working calendar and notebook 03's comparisons. The answer key is not used.

**Question:** Do the findings of notebooks 02 and 03 hold when reasonable measurement and population choices change — in particular how after-hours submissions are handled and whether requests unfinished at the extract are included? Sections A and B first look at unusually long and short requests and at which workflow measures move together.

**Reference analysis.** Notebooks 02 and 03 exactly as run: working time (Monday–Friday, 08:00–16:30, excluding the simulated closure dates), so an after-hours submission starts counting at the next working moment; the 4,319 completed requests; and the 13 contrasts with 95% intervals from 2,000 employee resamples (seed 2026), with notebook 03's weights, workload cutoffs and rules. Part 1 reproduces these numbers exactly before any alternative is computed.

**Alternatives, fixed before any of their results are computed.** In the cleaned records only submissions happen outside working hours (656 of 4,364, 15.0%); every later workflow event falls within them. Each alternative changes one kind of thing:

| Scenario | What changes | Kind of change |
|:---|:---|:---|
| H1 Elapsed calendar time | Time is counted from the actual submission, through nights, weekends and closures (the requester's wait), in calendar days | Measurement |
| H2 Weekdays-only calendar | Monday–Friday 08:00–16:30 as before, but closure dates count as working days | Calendar |
| H3 Working-hours submissions only | After-hours submissions are left out | Population |
| H4 Narrower after-hours label | Only weekend and closure-date submissions count as after-hours; weekday submissions outside 08:00–16:30 count as working-hours submissions | Classification (after-hours contrasts only) |
| U1 Unfinished, time so far | The 45 unfinished requests are added with their working time up to the extract (29 Sep 2026, 17:00), a lower bound on their eventual time | Population |
| U2 Unfinished, longest | The 45 are added as longer than every completed request | Population |
| O1 Without the long tail | The requests above notebook 02's Tukey fence (11.68 working-day equivalents) are left out | Population |

**Rules for the unfinished requests.** For percentiles of all requests (completed and unfinished), U1 and U2 are bounds: the eventual value lies between them. Under U2 a percentile that falls among the unfinished requests is reported as "above the longest completed request". For contrasts, U1 and U2 are sensitivity scenarios, not bounds, because the unfinished requests may be spread unevenly between the compared groups. Under U2 every unfinished request is given one value that exceeds both the longest completed duration and the longest time so far of any unfinished request (the larger of the two plus one working-day equivalent), so it is longer than every completed request and consistent with every unfinished request's observed lower bound, and medians stay defined. U1 stage times are working time accumulated up to the extract, a different quantity from the stage shares of completed requests.

**Rules for the contrasts.** Each scenario uses notebook 03's 13 contrasts, estimators and bootstrap (2,000 employee resamples, seed 2026, intervals withheld when more than 1% of resamples are invalid). Workload cutoffs stay at notebook 03's values; stratum weights are the strata's shares of the scenario's own requests. Because those weights follow the scenario's population, a weighted contrast that differs from the reference can reflect both changed results within strata and a changed mix of strata; each scenario's strata, weights and coverage are therefore reported. A contrast that a scenario removes (the after-hours contrasts under H3) is "not applicable"; one whose groups both exist but no stratum has at least five requests on each side is "not estimable", with no estimate or interval.

**What counts as a change, fixed now.** For each contrast, three things are reported separately: whether the point estimate changes direction, how much it changes, and whether the interval's relation to zero changes (excludes zero ↔ includes zero). The full task-type ordering is reported; a swap between task types whose reference medians differ by less than 0.25 working-day equivalents is reported as a small swap, separately from changes that affect a written finding. Every written finding of notebooks 02 and 03 then gets one label: **stable** (no change in any applicable scenario), **depends on scenario** (named), **not applicable** (the scenario removes it) or **not comparable** (H1 sizes are in calendar days, so only direction and ordering are compared). Intervals are descriptive: no p-values are calculated and no difference is called significant.

All results describe generated data; patterns are associations, not causes.

In [1]:
import hashlib
import json
from datetime import datetime, time, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")
FIGURE_DIR = Path("../figures")

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()

requests = read(SD / "cleaning/output/requests_cleaned.csv")
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())
assert sha(SD / "cleaning/output/requests_cleaned.csv") == cleaning_info["output_sha256"], \
    "The cleaned file doesn't match its cleaning record"

# Notebook 02's working calendar; the generator must be unchanged since then
calendar = json.loads((OUTPUT_DIR / "working_calendar.json").read_text())
assert sha(SD / "requests/generate_requests.py") == calendar["generator_sha256"], \
    "The generator has changed since notebook 02 saved the working calendar"
CLOSURES = {pd.Timestamp(d).date() for d in calendar["closure_dates"]}
DAY_START, DAY_END = time.fromisoformat(calendar["day_start"]), time.fromisoformat(calendar["day_end"])
DAY_HOURS = calendar["day_hours"]
EXTRACT_AT = pd.Timestamp(calendar["extract_at"])

# Inputs from notebooks 02 and 03 that this notebook must reproduce
inputs_03 = json.loads((OUTPUT_DIR / "analysis_inputs_03.json").read_text())
assert inputs_03["inputs_sha256"]["processing_times_completed.csv (notebook 02)"] == sha(OUTPUT_DIR / "processing_times_completed.csv")
assert inputs_03["inputs_sha256"]["working_calendar.json (notebook 02)"] == sha(OUTPUT_DIR / "working_calendar.json")
completed_02 = pd.read_csv(OUTPUT_DIR / "processing_times_completed.csv")
summary_02 = pd.read_csv(OUTPUT_DIR / "processing_time_summary.csv", index_col=0)
unfinished_02 = pd.read_csv(OUTPUT_DIR / "unfinished_requests.csv")
intervals_03 = pd.read_csv(OUTPUT_DIR / "contrast_intervals.csv")

TIME_COLUMNS = ["submitted_at", "info_complete_at", "assigned_at", "team_completed_at", "completion_confirmed_at"]
for c in TIME_COLUMNS:
    requests[c] = pd.to_datetime(requests[c].replace("", None), format="%Y-%m-%dT%H:%M")


def working_hours(start, end, closures=CLOSURES):
    # Notebook 02's working-time function, with the closure dates as a parameter (H2 passes an empty set)
    if pd.isna(start) or pd.isna(end):
        return np.nan
    if end < start:
        raise ValueError(f"End precedes start: {start} → {end}")
    if end == start:
        return 0.0
    total, day = 0.0, start.date()
    while day <= end.date():
        if day.weekday() < 5 and day not in closures:
            open_, close = datetime.combine(day, DAY_START), datetime.combine(day, DAY_END)
            total += max((min(end, close) - max(start, open_)).total_seconds(), 0) / 3600
        day += timedelta(days=1)
    return total


# Notebook 02's hand-made tests, plus the same closure case with closures counted as working days (H2)
T = pd.Timestamp
tests = pd.DataFrame([
    ("same working day", T("2026-03-03 09:00"), T("2026-03-03 11:30"), CLOSURES, 2.5),
    ("overnight", T("2026-03-03 15:30"), T("2026-03-04 09:00"), CLOSURES, 2.0),
    ("over a weekend", T("2026-03-06 16:00"), T("2026-03-09 08:30"), CLOSURES, 1.0),
    ("after-hours submission", T("2026-03-06 21:00"), T("2026-03-09 10:00"), CLOSURES, 2.0),
    ("over a simulated closure date (Fri 3 Apr 2026)", T("2026-04-02 16:00"), T("2026-04-06 09:00"), CLOSURES, 1.5),
    ("the same, closure counted as a working day (H2)", T("2026-04-02 16:00"), T("2026-04-06 09:00"), set(), 10.0),
    ("whole working day", T("2026-03-03 08:00"), T("2026-03-03 16:30"), CLOSURES, DAY_HOURS),
    ("start equals end", T("2026-03-03 10:00"), T("2026-03-03 10:00"), CLOSURES, 0.0),
], columns=["case", "start", "end", "closures", "expected_hours"])
assert T("2026-04-03").date() in CLOSURES
tests["calculated_hours"] = [working_hours(s, e, c) for s, e, c in zip(tests["start"], tests["end"], tests["closures"])]
assert np.isclose(tests["calculated_hours"], tests["expected_hours"]).all(), "The working-time function failed a test"

# Every workflow event other than submission falls inside working hours, as this notebook's design assumes
def outside_hours(t):
    t = t.dropna()
    return int(((t.dt.weekday >= 5) | t.dt.date.isin(CLOSURES) | ~t.dt.time.between(DAY_START, DAY_END, inclusive="both")).sum())
after_hours_events = {c: outside_hours(requests[c]) for c in TIME_COLUMNS}
assert all(v == 0 for c, v in after_hours_events.items() if c != "submitted_at"), "A later workflow event falls outside working hours"

print(f"Data version {DATA_VERSION}; cleaning version {cleaning_info['cleaning_version']}; extract {EXTRACT_AT:%Y-%m-%d %H:%M}")
print(f"{len(requests):,} cleaned requests; working-time function passes {len(tests)} hand-made tests (including H2's calendar)")
print("Events outside working hours, by timestamp:", after_hours_events)

Data version 2; cleaning version 1.1; extract 2026-09-29 17:00
4,364 cleaned requests; working-time function passes 8 hand-made tests (including H2's calendar)
Events outside working hours, by timestamp: {'submitted_at': 656, 'info_complete_at': 0, 'assigned_at': 0, 'team_completed_at': 0, 'completion_confirmed_at': 0}


## The reference, reproduced

The cell below rebuilds every request's durations and notebook 03's variables from the cleaned requests, using notebook 02's and notebook 03's rules unchanged: stage times and processing time in working-day equivalents, elapsed calendar days, working time so far for the unfinished requests, workload at assignment and its groups, the after-hours label and the intake channel. It then rebuilds notebook 03's 13 contrasts and their employee-bootstrap intervals, written as functions so every scenario can reuse them.

Before anything else is computed, the rebuilt values must match the saved outputs of notebooks 02 and 03, which the later sections use as the reference:

- each completed request's processing time, stage times and calendar days, and each unfinished request's time so far;
- notebook 02's overall summaries in working-day equivalents and calendar days (median, mean, quartiles, IQR, 90th percentile, longest, skew, Tukey fence and the number and share above it), its summaries by task type, its stage summary and its stage shares by task type;
- every contrast's point estimate, interval, number of valid resamples and whether its interval was withheld.

Counts must match exactly; other values are compared to within 1e-9, to allow only for floating-point rounding.

In [2]:
STAGES = ["info_check_h", "assignment_wait_h", "team_processing_h"]
TASK_ORDER = ["payroll_setup", "system_access", "workplace_access", "equipment_setup", "orientation_scheduling"]
r = requests.copy()
finished = r["team_completed_at"].notna()
all_stage_times = r[["submitted_at", "info_complete_at", "assigned_at", "team_completed_at"]].notna().all(axis=1)
r["group"] = np.select([finished & all_stage_times, ~finished], ["completed", "unfinished at extract"], default="other")
assert (r["group"] != "other").all(), "Every request should be completed with all stage times, or unfinished"


def durations(frame, closures=CLOSURES):
    # Stage, processing and time-so-far working durations under a closure calendar, in hours and working-day equivalents
    out = pd.DataFrame(index=frame.index)
    out["info_check_h"] = [working_hours(a, b, closures) for a, b in zip(frame["submitted_at"], frame["info_complete_at"])]
    out["assignment_wait_h"] = [working_hours(a, b, closures) for a, b in zip(frame["info_complete_at"], frame["assigned_at"])]
    out["team_processing_h"] = [working_hours(a, b, closures) for a, b in zip(frame["assigned_at"], frame["team_completed_at"])]
    out["processing_days"] = [working_hours(a, b, closures) / DAY_HOURS for a, b in zip(frame["submitted_at"], frame["team_completed_at"])]
    out["days_so_far"] = [working_hours(a, EXTRACT_AT, closures) / DAY_HOURS for a in frame["submitted_at"]]
    out["elapsed_calendar_days"] = (frame["team_completed_at"] - frame["submitted_at"]).dt.total_seconds() / 86400
    out["calendar_days_so_far"] = (EXTRACT_AT - frame["submitted_at"]).dt.total_seconds() / 86400
    return out


r = r.join(durations(r))
r["processing_h"] = r["processing_days"] * DAY_HOURS
STAGE_NAMES = {"info_check_h": "information check", "assignment_wait_h": "assignment wait", "team_processing_h": "team processing"}


def describe(values):
    # Notebook 02's summary of a duration
    values = values.dropna()
    q1, med, q3, p90 = values.quantile([0.25, 0.5, 0.75, 0.9])
    fence = q3 + 1.5 * (q3 - q1)
    return pd.Series({"requests": int(values.count()), "median": med, "mean": values.mean(), "q1": q1, "q3": q3,
                      "iqr": q3 - q1, "p90": p90, "max": values.max(), "skew": values.skew(), "tukey_fence": fence,
                      "unusually_long": int((values > fence).sum()), "unusually_long_%": 100 * (values > fence).mean()})


def stage_table(frame):
    # Notebook 02's stage summary: median and mean stage time, and each stage's share of total working time
    total = frame["processing_h"].sum()
    rows = {name: {"requests": len(frame), "median_days": (frame[col] / DAY_HOURS).median(),
                   "mean_days": (frame[col] / DAY_HOURS).mean(), "share_of_total_%": 100 * frame[col].sum() / total}
            for col, name in STAGE_NAMES.items()}
    return pd.DataFrame(rows).T


# Notebook 03's variables, unchanged
assigned = r[r["assigned_at"].notna()]
workload = {}
for team, g in assigned.groupby("responsible_team"):
    a = g["assigned_at"].to_numpy()
    done = g["team_completed_at"].fillna(pd.Timestamp.max).to_numpy()
    for rid, t in zip(g["request_id"], a):
        workload[rid] = int(((a < t) & (done > t)).sum())
r["workload_at_assignment"] = r["request_id"].map(workload)
r["intake_channel"] = r["intake_channel"].replace("", "unknown")
r["info_issue_at_intake"] = r["info_issue_at_intake"].map({"True": True, "False": False})
s = r["submitted_at"]
in_hours = s.dt.time.between(DAY_START, DAY_END, inclusive="both")
r["after_hours"] = (s.dt.weekday >= 5) | s.dt.date.isin(CLOSURES) | ~in_hours
r["after_hours_narrow"] = (s.dt.weekday >= 5) | s.dt.date.isin(CLOSURES)           # H4's label

# The completed requests, in notebook 02's row order (the bootstrap draws depend on it)
completed = r.set_index("request_id").loc[completed_02["request_id"]].reset_index()
Q1, Q3 = completed["workload_at_assignment"].quantile([0.25, 0.75])                # notebook 03's cutoffs, kept for every scenario
group_workload = lambda w: np.select([w <= Q1, w >= Q3], ["low", "high"], default="middle").astype(object)
r["workload_group"] = np.where(r["workload_at_assignment"].isna(), "not assigned", group_workload(r["workload_at_assignment"]))
completed["workload_group"] = group_workload(completed["workload_at_assignment"])
unfinished = r[r["group"] == "unfinished at extract"].copy()

# Reproduction of notebook 02
close = lambda a, b: np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), rtol=0, atol=1e-9)
REPRO_02 = (completed["request_id"].tolist() == completed_02["request_id"].tolist()
            and all(close(completed[c], completed_02[c]) for c in ["processing_days", "elapsed_calendar_days"] + STAGES)
            and (completed["task_type"].to_numpy() == completed_02["task_type"].to_numpy()).all()
            and set(unfinished["request_id"]) == set(unfinished_02["request_id"])
            and close(unfinished.set_index("request_id").loc[unfinished_02["request_id"], "days_so_far"], unfinished_02["days_so_far"]))
ref_summary = pd.DataFrame({"working-day equivalents": describe(completed["processing_days"]),
                            "elapsed calendar days": describe(completed["elapsed_calendar_days"])}).T
ref_by_task = completed.groupby("task_type")["processing_days"].apply(describe).unstack().reindex(TASK_ORDER)
ref_stages = stage_table(completed)
ref_shares_by_task = pd.DataFrame({t: stage_table(completed[completed["task_type"] == t])["share_of_total_%"] for t in TASK_ORDER}).T
by_task_02 = pd.read_csv(OUTPUT_DIR / "processing_time_by_task_type.csv", index_col=0)
stages_02 = pd.read_csv(OUTPUT_DIR / "stage_summary.csv", index_col=0)
shares_02 = pd.read_csv(OUTPUT_DIR / "stage_shares_by_task_type.csv", index_col=0)
COUNTS = ["requests", "unusually_long"]
REPRO_02 = REPRO_02 and (
    (ref_summary[COUNTS].astype(int).to_numpy() == summary_02[COUNTS].to_numpy()).all()
    and close(ref_summary.drop(columns=COUNTS), summary_02[ref_summary.drop(columns=COUNTS).columns])
    and (ref_by_task["requests"].astype(int).to_numpy() == by_task_02["requests"].to_numpy()).all()
    and close(ref_by_task[["median", "mean", "q1", "q3", "iqr", "p90", "max", "skew"]],
              by_task_02[["median", "mean", "q1", "q3", "iqr", "p90", "max", "skew"]])
    and (ref_stages["requests"].astype(int).to_numpy() == stages_02["requests"].to_numpy()).all()
    and close(ref_stages[["median_days", "mean_days", "share_of_total_%"]], stages_02[["median_days", "mean_days", "share_of_total_%"]])
    and close(ref_shares_by_task, shares_02[list(STAGE_NAMES.values())]))
FENCE = ref_summary.loc["working-day equivalents", "tukey_fence"]
assert REPRO_02, "The rebuilt durations or summaries do not reproduce notebook 02"

# Notebook 03's contrasts and bootstrap, as functions of a scenario's requests and times
MIN_PER_SIDE, B, MAX_INVALID_SHARE, SEED = 5, 2000, 0.01, 2026
TASKS = ["equipment_setup", "system_access", "workplace_access", "orientation_scheduling"]
CONTRASTS = (
    [(f"{t} - payroll_setup", "task_type", t, "payroll_setup", None) for t in TASKS]
    + [("information issue - none", "info_issue_at_intake", True, False, None),
       ("high - low workload", "workload_group", "high", "low", None),
       ("high - low workload, within task type", "workload_group", "high", "low", ["task_type"]),
       ("after-hours - working hours", "after_hours", True, False, None),
       ("after-hours - working hours, within information issue x task type", "after_hours", True, False,
        ["info_issue_at_intake", "task_type"]),
       ("phone - web form", "intake_channel", "phone", "web_form", None),
       ("phone - web form, within information issue", "intake_channel", "phone", "web_form", ["info_issue_at_intake"]),
       ("email - web form", "intake_channel", "email", "web_form", None),
       ("email - web form, within information issue", "intake_channel", "email", "web_form", ["info_issue_at_intake"])])


def make_plans(df):
    # Notebook 03's contrast plans on one scenario's requests: groups, supported strata and weights
    plans = []
    for label, col, comp, ref, strata in CONTRASTS:
        comp_mask, ref_mask = (df[col] == comp).to_numpy(), (df[col] == ref).to_numpy()
        applicable = bool(comp_mask.any() and ref_mask.any())
        plan = {"label": label, "comp": comp_mask, "ref": ref_mask, "strata": None, "applicable": applicable,
                "reason": "" if applicable else "a compared group is empty in this scenario"}
        if strata and applicable:
            key = df[strata].astype(str).agg(" | ".join, axis=1).to_numpy()
            cells = [(k, (key == k).sum() / len(df)) for k in sorted(set(key))
                     if (comp_mask & (key == k)).sum() >= MIN_PER_SIDE and (ref_mask & (key == k)).sum() >= MIN_PER_SIDE]
            if not cells:                                            # both groups exist, but no stratum supports a comparison
                plan.update({"applicable": False, "reason": f"no stratum has at least {MIN_PER_SIDE} requests on each side"})
            else:
                weights = np.array([w for _, w in cells], dtype=float)
                plan.update({"strata": [k for k, _ in cells], "key": key, "weights": weights / weights.sum(),
                             "coverage_%": 100 * weights.sum(), "stratum_weights": dict(zip([k for k, _ in cells], weights / weights.sum()))})
        plans.append(plan)
    return plans


def estimate(plan, y, idx):
    yy, comp, ref = y[idx], plan["comp"][idx], plan["ref"][idx]
    if plan["strata"] is None:
        if not comp.any() or not ref.any():
            return np.nan
        return np.median(yy[comp]) - np.median(yy[ref])
    key = plan["key"][idx]
    diffs = []
    for k in plan["strata"]:
        c, r_ = comp & (key == k), ref & (key == k)
        if not c.any() or not r_.any():
            return np.nan
        diffs.append(np.median(yy[c]) - np.median(yy[r_]))
    return float(np.dot(plan["weights"], diffs))


def contrast_table(df, y):
    # Point estimates and 95% employee-bootstrap intervals for the 13 contrasts on one scenario
    plans = make_plans(df)
    emp_rows = df.groupby("employee_id").indices
    employees = np.array(sorted(emp_rows))
    rng = np.random.default_rng(SEED)
    draws = np.full((B, len(plans)), np.nan)
    for b in range(B):
        idx = np.concatenate([emp_rows[e] for e in rng.choice(employees, size=len(employees), replace=True)])
        draws[b] = [estimate(p, y, idx) if p["applicable"] else np.nan for p in plans]
    rows, all_idx = [], np.arange(len(df))
    for j, p in enumerate(plans):
        valid = draws[np.isfinite(draws[:, j]), j]
        withheld = not p["applicable"] or (B - len(valid)) / B > MAX_INVALID_SHARE
        rows.append({"contrast": p["label"], "applicable": p["applicable"], "reason": p["reason"],
                     "estimate_days": estimate(p, y, all_idx) if p["applicable"] else np.nan,
                     "ci_low": np.nan if withheld else np.percentile(valid, 2.5),
                     "ci_high": np.nan if withheld else np.percentile(valid, 97.5),
                     "valid_resamples": len(valid), "interval_withheld": withheld,
                     "comparison_requests": int(p["comp"].sum()), "reference_requests": int(p["ref"].sum()),
                     "coverage_%": p.get("coverage_%", np.nan)})
    return pd.DataFrame(rows), plans


# Self-test: both groups exist, but no stratum has enough requests on each side, so the contrast is not estimable (not 0)
_toy = pd.DataFrame({"employee_id": [f"E{i}" for i in range(8)], "task_type": ["payroll_setup"] * 4 + ["system_access"] * 4,
                     "info_issue_at_intake": [True, False] * 4, "workload_group": ["low", "high"] * 4,
                     "after_hours": [True, False] * 4, "intake_channel": ["phone", "web_form", "email", "web_form"] * 2})
_toy_plan = next(p for p in make_plans(_toy) if p["label"] == "high - low workload, within task type")
assert not _toy_plan["applicable"] and _toy_plan["reason"].startswith("no stratum"), "An unsupported weighted contrast must be not estimable"

reference, reference_plans = contrast_table(completed, completed["processing_days"].to_numpy())
REPRO_03 = (reference["contrast"].tolist() == intervals_03["contrast"].tolist()
            and all(close(reference[c], intervals_03[c]) for c in ["estimate_days", "ci_low", "ci_high"])
            and (reference["valid_resamples"].to_numpy() == intervals_03["valid_resamples"].to_numpy()).all()
            and (reference["interval_withheld"].to_numpy() == intervals_03["interval_withheld"].to_numpy()).all())
assert REPRO_03, "The rebuilt contrasts do not reproduce notebook 03"
print(f"Reference reproduced: {len(completed):,} completed requests (median {completed['processing_days'].median():.2f} "
      f"working-day equivalents, fence {FENCE:.2f}), {len(unfinished)} unfinished requests, and notebook 03's 13 contrasts "
      f"and intervals (workload cutoffs {Q1:g} and {Q3:g})")
display(reference.set_index("contrast")[["estimate_days", "ci_low", "ci_high", "comparison_requests", "reference_requests"]].round(2))

Reference reproduced: 4,319 completed requests (median 4.66 working-day equivalents, fence 11.68), 45 unfinished requests, and notebook 03's 13 contrasts and intervals (workload cutoffs 9 and 22)


,estimate_days,ci_low,ci_high,comparison_requests,reference_requests
contrast,,,,,
equipment_setup - payroll_setup,0.66,0.37,0.91,632,1061
system_access - payroll_setup,-0.81,-1.06,-0.59,1059,1061
workplace_access - payroll_setup,-0.19,-0.51,0.02,666,1061
orientation_scheduling - payroll_setup,-1.17,-1.45,-0.93,901,1061
information issue - none,2.80,2.54,3.12,563,3756
high - low workload,0.59,0.30,0.84,1132,1160
"high - low workload, within task type",0.09,-0.44,0.46,1132,1160
after-hours - working hours,0.23,0.03,0.55,650,3669
"after-hours - working hours, within information issue x task type",0.34,0.12,0.54,650,3669
